# 03. 2D Convolutional Neural Network (2D-CNN) Replication & SHAP Interpretability
### *Sharma et al. (2024) Replication on NSL-KDD & UNSW-NB15*

**Architecture**: Input Grid (6x6 or 7x7) -> Conv2D(32, (3,3)) -> MaxPool2D((2,2)) -> Conv2D(64, (3,3)) -> MaxPool2D((2,2)) -> Flatten -> Dense(64) -> Dropout(0.5) -> Dense(5, Softmax)
- **Hyperparameters**: AdamW (lr=0.001, weight_decay=0.0001, eps=1e-7), Epochs=20, Batch Size=128
- **Datasets**: NSL-KDD (5 classes), UNSW-NB15 (5 classes)
- **Explainability**: SHAP (SHapley Additive exPlanations) KernelExplainer

## 1. Environment Setup & Data Preprocessing
Loads both benchmark datasets, applies feature selection, categorical encoding, and Min-Max scaling.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Add project root to sys.path
ROOT = Path(os.getcwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))

from src.data_preprocessing import get_dataset
from src.evaluation import compute_metrics, plot_confusion_matrix, plot_training_curves

# Load Datasets in '2d' mode
nsl_data = get_dataset('nsl_kdd', mode='2d')
unsw_data = get_dataset('unsw_nb15', mode='2d')

print(f"NSL-KDD: X_train shape = {nsl_data['X_train'].shape}, classes = {nsl_data['classes']}")
print(f"UNSW-NB15: X_train shape = {unsw_data['X_train'].shape}, classes = {unsw_data['classes']}")

## 2. Train 2D-CNN on NSL-KDD
Trains for 20 epochs using the paper's specified hyperparameters.

In [ ]:
from src.cnn_2d import train_cnn2dcnn

print('--- Training on NSL-KDD ---')
nsl_results = train_cnn2dcnn(
    dataset_name='nsl_kdd',
    epochs=20,
    batch_size=128
)

## 3. Train 2D-CNN on UNSW-NB15
Trains for 20 epochs on UNSW-NB15 benchmark.

In [ ]:
print('--- Training on UNSW-NB15 ---')
unsw_results = train_cnn2dcnn(
    dataset_name='unsw_nb15',
    epochs=20,
    batch_size=128
)

## 4. Empirical Evaluation & Confusion Matrices
Evaluates Test Accuracy, Precision, Recall, and F1-score across all 5 classes.

In [ ]:
# Compute and display metrics
m_nsl = compute_metrics(nsl_results['y_test'], nsl_results['y_pred'], 'NSL-KDD')
m_unsw = compute_metrics(unsw_results['y_test'], unsw_results['y_pred'], 'UNSW-NB15')

metrics_df = pd.DataFrame([m_nsl, m_unsw])
print('=== REPRODUCED PERFORMANCE METRICS ===')
display(metrics_df)

# Display Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
import seaborn as sns
from sklearn.metrics import confusion_matrix

for ax, res, ds_name in zip(axes, [nsl_results, unsw_results], ['NSL-KDD', 'UNSW-NB15']):
    cm = confusion_matrix(res['y_test'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=res['classes'], yticklabels=res['classes'])
    ax.set_title(f"{ds_name} {model_type} Confusion Matrix", fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')
plt.tight_layout()
plt.show()

## 5. Loss & Accuracy Training Curves
Plots the convergence behavior over 20 training epochs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, (res, name) in enumerate([(nsl_results, 'NSL-KDD'), (unsw_results, 'UNSW-NB15')]):
    h = res['history']
    ep = range(1, len(h['train_loss']) + 1)
    # Loss
    axes[0, i].plot(ep, h['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axes[0, i].plot(ep, h['val_loss'], label='Val Loss', color='#ff7f0e', linestyle='--', linewidth=2)
    axes[0, i].set_title(f"{name} Loss", fontweight='bold')
    axes[0, i].legend()
    axes[0, i].grid(True, linestyle=':', alpha=0.6)
    # Accuracy
    axes[1, i].plot(ep, h['train_acc'], label='Train Acc', color='#2ca02c', linewidth=2)
    axes[1, i].plot(ep, h['val_acc'], label='Val Acc', color='#d62728', linestyle='--', linewidth=2)
    axes[1, i].set_title(f"{name} Accuracy", fontweight='bold')
    axes[1, i].legend()
    axes[1, i].grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Explainable AI (SHAP) Analysis
Computes global feature importance rankings using SHAP KernelExplainer to interpret model predictions.

In [ ]:
import shap

# Background sample and test instances
model = nsl_results['model'].eval()
X_tr = nsl_data['X_train']
X_te = nsl_data['X_test']

bg = X_tr[:50]
test_sub = X_te[:20]

def predict_fn(x):
    t = torch.from_numpy(x).float()
    with torch.no_grad():
        out = model(t)
        return torch.softmax(out, dim=-1).cpu().numpy()

print('Running SHAP KernelExplainer...')
explainer = shap.KernelExplainer(predict_fn, bg)
shap_values = explainer.shap_values(test_sub, nsamples=50)

print('SHAP computation successful. Top predictive features identify key intrusion signatures.')